In [1]:
# ==============================================================================
# 1. CÀI ĐẶT & IMPORT
# ==============================================================================
print("⏳ Đang cài đặt FinRL...")
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q
!pip install shimmy>=0.2.1 -q
!pip install pyportfolioopt -q # Thư viện tối ưu danh mục

import pandas as pd
import numpy as np
import datetime
import os
import matplotlib.pyplot as plt
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config
from finrl.config import INDICATORS

# Import Môi trường Portfolio (Khác môi trường Trading cũ)
from finrl.meta.env_portfolio_allocation.env_portfolio import StockPortfolioEnv
from finrl.agents.stablebaselines3.models import DRLAgent

# Tạo thư mục lưu model
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)

print("✅ Cài đặt xong!")

⏳ Đang cài đặt FinRL...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/108.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 73.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Cài đặt xong!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# ==============================================================================
# 2. TẢI DỮ LIỆU & XỬ LÝ (ĐÃ FIX LỖI ATTRIBUTE ERROR)
# ==============================================================================
# Khai báo list ticker thủ công để tránh lỗi config cũ
ticker_list = [
    "AXP", "AMGN", "AAPL", "BA", "CAT", "CSCO", "CVX", "GS", "HD", "HON",
    "IBM", "INTC", "JNJ", "KO", "JPM", "MCD", "MMM", "MRK", "MSFT", "NKE",
    "PG", "TRV", "UNH", "CRM", "VZ", "V", "WBA", "WMT", "DIS", "DOW"
]

print("📥 Đang tải dữ liệu Dow 30 (2009 - 2025)...")
try:
    df = YahooDownloader(start_date = '2009-01-01',
                         end_date = '2025-12-12',
                         ticker_list = ticker_list).fetch_data()
except Exception as e:
    print(f"⚠️ Có lỗi nhỏ khi tải data: {e}. Đang thử tải lại...")
    df = YahooDownloader(start_date = '2009-01-01',
                         end_date = '2025-12-12',
                         ticker_list = ticker_list).fetch_data()

print("⚙️ Đang xử lý chỉ báo kỹ thuật...")
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list=INDICATORS,
                    use_vix=True,
                    use_turbulence=True,
                    user_defined_feature = False)

df = fe.preprocess_data(df)

# --- TÍNH TOÁN COVARIANCE MATRIX ---
print("🧮 Đang tính toán Ma trận Hiệp phương sai (Covariance)...")

# Sắp xếp tạm để tính toán
df = df.sort_values(['date','tic'], ignore_index=True)
df.index = df.date.factorize()[0]

cov_list = []
return_list = []
lookback = 252 # 1 năm

unique_dates = df.date.unique()

# Vòng lặp tính toán
for i in range(lookback, len(unique_dates)):
    data_lookback = df.loc[i-lookback:i, :]
    price_lookback = data_lookback.pivot_table(index='date', columns='tic', values='close')
    return_lookback = price_lookback.pct_change().dropna()

    return_list.append(return_lookback.values)
    cov_list.append(return_lookback.cov().values)

# Ghép Covariance vào data
df_cov = pd.DataFrame({
    'date': unique_dates[lookback:],
    'cov_list': cov_list,
    'return_list': return_list
})

df = df.merge(df_cov, on='date')
df = df.sort_values(['date','tic']).reset_index(drop=True)

# --- BƯỚC SỬA LỖI QUAN TRỌNG TẠI ĐÂY ---
# Môi trường Portfolio bắt buộc Index phải là thứ tự ngày (0,0,0... 1,1,1...)
# Lệnh dưới đây sẽ gán lại index chuẩn cho StockPortfolioEnv
df.index = df['date'].factorize()[0]
# ---------------------------------------

# Tạo tập Train
df_train = df.copy()

print(f"✅ Dữ liệu huấn luyện đã sẵn sàng: {df_train.shape}")
print(f"👉 Kiểm tra Index dòng đầu: {df_train.index[0]}")
print(f"👉 Kiểm tra Index dòng cuối: {df_train.index[-1]}")
print("   (Nếu Index dòng đầu = 0 và dòng cuối là số lớn, code đã đúng!)")

[*********************100%***********************]  1 of 1 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

[*********************100%***********************]  1 of 1 completed

📥 Đang tải dữ liệu Dow 30 (2009 - 2025)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[*********************100%***********************]  1 of 1 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return dateti

Shape of DataFrame:  (121058, 8)
⚙️ Đang xử lý chỉ báo kỹ thuật...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Successfully added turbulence index
🧮 Đang tính toán Ma trận Hiệp phương sai (Covariance)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Dữ liệu huấn luyện đã sẵn sàng: (112280, 20)
👉 Kiểm tra Index dòng đầu: 0
👉 Kiểm tra Index dòng cuối: 4009
   (Nếu Index dòng đầu = 0 và dòng cuối là số lớn, code đã đúng!)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
# ==============================================================================
# 3. THIẾT LẬP MÔI TRƯỜNG & HUẤN LUYỆN (ĐÃ FIX LỖI RESULTS)
# ==============================================================================
import os

# --- BƯỚC SỬA LỖI QUAN TRỌNG ---
# Tạo thư mục 'results' để môi trường lưu biểu đồ, tránh lỗi FileNotFoundError
if not os.path.exists("results"):
    os.makedirs("results")
print("✅ Đã tạo thư mục 'results' để lưu biểu đồ.")
# -------------------------------

stock_dimension = len(df_train.tic.unique())
state_space = stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "transaction_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

e_train_gym = StockPortfolioEnv(df = df_train, **env_kwargs)
agent = DRLAgent(env = e_train_gym)

# --- TRAIN A2C ---
print("\n🚀 Đang huấn luyện A2C (Portfolio Allocation)...")
model_a2c = agent.get_model("a2c")
trained_a2c = agent.train_model(model=model_a2c,
                             tb_log_name="a2c",
                             total_timesteps=30000)
trained_a2c.save("trained_models/policy_a2c")

# --- TRAIN PPO ---
print("\n🚀 Đang huấn luyện PPO (Portfolio Allocation)...")
model_ppo = agent.get_model("ppo")
trained_ppo = agent.train_model(model=model_ppo,
                             tb_log_name="ppo",
                             total_timesteps=30000)
trained_ppo.save("trained_models/policy_ppo")

# --- ĐÓNG GÓI ---
print("\n🎉 HUẤN LUYỆN HOÀN TẤT!")
import shutil
shutil.make_archive("portfolio_models", 'zip', "trained_models")
print("👉 File 'portfolio_models.zip' đã sẵn sàng. Hãy tải về máy!")

✅ Đã tạo thư mục 'results' để lưu biểu đồ.
Stock Dimension: 28, State Space: 28

🚀 Đang huấn luyện A2C (Portfolio Allocation)...
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-------------------------------------
| time/                 |           |
|    fps                | 253       |
|    iterations         | 100       |
|    time_elapsed       | 1         |
|    total_timesteps    | 500       |
| train/                |           |
|    entropy_loss       | -39.4     |
|    explained_variance | -2.38e-07 |
|    learning_rate      | 0.0007    |
|    n_updates          | 99        |
|    policy_loss        | 1.42e+08  |
|    reward             | 1282810.8 |
|    reward_max         | 1282810.8 |
|    reward_mean        | 1270133.4 |
|    reward_min         | 1259756.0 |
|    std                | 0.987     |
|    value_loss         | 1.62e+13  |
-----------------------------------